# SS3 Part B - Applying the Final SS2 Method to Client Data

**Client:** STADIOEquities 
**Task:** Fraud detection & AML Alert triage
**Method (from SS2):** Random Forest classifier with SMOTE on the trianing set only, evaluated with imbalance-aware metrics

---
## Component 1: Dataset Loading 


In [4]:
import pandas as pd
import numpy as np

# Load the core labelled table - this table contains the target (investigation_disposition)
alerts = pd.read_csv("../data/raw_ss3/table1_compliance_alerts.csv")

# The target has three classes; check the distribution
print("Total alerts loaded:", len(alerts))
print("\nTarget (investigation_disposition) distribution:")
print(alerts["investigation_disposition"].value_counts())
print("\nProportions:")
print(alerts["investigation_disposition"].value_counts(normalize=True).round(4))

Total alerts loaded: 2864

Target (investigation_disposition) distribution:
investigation_disposition
False positive    2236
True positive      351
Unresolved         277
Name: count, dtype: int64

Proportions:
investigation_disposition
False positive    0.7807
True positive     0.1226
Unresolved        0.0967
Name: proportion, dtype: float64


### Dataset loading — explanation

**Source and format.** The data is loaded from `table1_compliance_alerts.csv`, the
core labelled table from the client extract. It is supplied as CSV, one row per
compliance alert, which is the natural format for this tabular, row-per-record
problem and matches how the client's monitoring system exports alerts. The other
six tables in the extract provide supporting context and are joined in during
preprocessing; this table is loaded first because it is the only one containing
the target label.

**Number of instances.** There are **2,864 alerts** in total.

**Classes.** The target, `investigation_disposition`, has three values:

| Class | Count | Proportion |
|---|---|---|
| False positive | 2,236 | 78.1% |
| True positive | 351 | 12.3% |
| Unresolved | 277 | 9.7% |

**Imbalance.** The problem is imbalanced. Once the 277 **Unresolved** alerts are
removed (they carry no known outcome and cannot be used as training labels), the
task becomes a **binary classification** of False positive versus True positive,
with roughly **86% False positive and 14% True positive**. This imbalance is
central to the project: the dominant class is false positives, which is precisely
the burden the triage model is intended to reduce. It also means accuracy is an
unsuitable headline metric, and precision, recall, F1 and PR-AUC are used instead,
consistent with the SS2 method.

---

## Component 2: Dataset Preprocessing

In [5]:
# Step 2 a: Remove Unresolved alerts and build the binary target
# Unresolved alerts have no known outcome, so they cannot be used as training
# labels. We remove them, leaving a binary problem: True positive vs False positive

# Will work on a copy
df = alerts.copy()

# Clean the target text (strip stray whitespace) then filter out Unresolved
df["investigation_disposition"] = df["investigation_disposition"].str.strip()
df = df[df["investigation_disposition"] != "Unresolved"].copy()

# Create the binary target: 1 = True positive (genuine case), 0 = False positive
df["target"] = (df["investigation_disposition"] == "True positive").astype(int)

print ("Rows after removing Unresolved:", len(df))
print("\nBinary target distribution:")
print(df["target"].value_counts())
print("\nProportion positive (True positive):", round(df["target"].mean(), 4))

Rows after removing Unresolved: 2587

Binary target distribution:
target
0    2236
1     351
Name: count, dtype: int64

Proportion positive (True positive): 0.1357


In [6]:
# Step 2b: Clean messy categorical text fields
# The extract has inconsistent casing and stray whitespace (e.g. 'GP', 'gp', ' GP ').
# We standardise the text fields by stripping whitespace and fixing casing so that 
# the same real value is not treated as several different categories

def clean_text(series):
    # Strip whitespace; leave casing fix to per-field handling below
    return series.astype(str).str.strip()

# Clean the alert-level categorical we will use as a feature
df["rule_triggered"] = clean_text(df["rule_triggered"])

print("rule_triggered categories after cleaning:")
print(df["rule_triggered"].value_counts())



rule_triggered categories after cleaning:
rule_triggered
Large withdrawal              419
Rapid deposit-withdrawal      344
High-frequency trading        319
New account high value        278
Round-amount pattern          224
Structuring pattern           222
Third-party funding           213
Dormant reactivation          195
Offshore transfer velocity    192
Other                         181
Name: count, dtype: int64


In [7]:
## Step 2c: Join features from the account, demographics and transaction tables
# The alert table alone has limited features. We strengthen each alert with:
# - account profile (KYC status, account status, premium) from table 2
# - demographics (age, province, risk appetite, goal) from table 3
# - transaction context (type, amount, concentration) from table 4, where the 
# alert is linked to a specified transaction.

# Load the supporting tables
accounts = pd.read_csv("../data/raw_ss3/table2_account_funding_profile.csv")
demographics = pd.read_csv("../data/raw_ss3/table3_client_demographics.csv")
transactions = pd.read_csv("../data/raw_ss3/table4_transaction_history.csv")

# Table 2 has 9 duplicate account_ids -  drop duplicates so the join is one to one
accounts = accounts.drop_duplicates(subset="account_id", keep="first")

# Join account profile and demographics on account_id
df = df.merge(accounts[["account_id", "kyc_status", "account_status",
                        "premium_subscription_status"]],
              on="account_id", how="left")
df = df.merge(demographics[["account_id", "age", "province_region",
                            "stated_risk_appetite", "stated_investment_goal"]],
              on="account_id", how="left")

# Join transaction context on transaction_id (only ~58% of alerts have one)
df = df.merge(transactions[["transaction_id", "transaction_type", "amount",
                            "portfolio_concentration_at_transaction"]],
              on="transaction_id", how="left")

print("Shape after joins:", df.shape)
print("\nColumns now available:")
print(list(df.columns))

Shape after joins: (2587, 21)

Columns now available:
['alert_id', 'alert_timestamp', 'rule_triggered', 'account_id', 'transaction_id', 'rule_assigned_severity_score', 'investigation_disposition', 'case_closed_timestamp', 'investigator_notes_anonymised', 'assigned_investigator_id_anonymised', 'target', 'kyc_status', 'account_status', 'premium_subscription_status', 'age', 'province_region', 'stated_risk_appetite', 'stated_investment_goal', 'transaction_type', 'amount', 'portfolio_concentration_at_transaction']


In [8]:
# Step 2d: Select features, clean remaining issues, handle missing values

# Choose the feature columns for the model (drop IDs, timestamps, text notes,
# and leakage-risk columns like the disposition text itself)
feature_cols = [
    "rule_triggered", "rule_assigned_severity_score",
    "kyc_status", "account_status", "premium_subscription_status",
    "age", "province_region", "stated_risk_appetite", "stated_investment_goal",
    "transaction_type", "amount", "portfolio_concentration_at_transaction",
]

model_df = df[feature_cols + ["target"]].copy()

# Clean whitespace/casing on the joined categorical text fields
cat_text_cols = ["kyc_status", "account_status", "premium_subscription_status",
                 "province_region", "stated_risk_appetite", "stated_investment_goal",
                 "transaction_type"]
for col in cat_text_cols:
    model_df[col] = model_df[col].astype(str).str.strip()
    # standardise province casing specifically (had 'gp', ' GP ', etc.)
model_df["province_region"] = model_df["province_region"].str.upper()

# Fix out of range ages (as README warned data capture errors).
# Treat implausible ages (over 100 or under 18) as missing.
model_df.loc[(model_df["age"] > 100) | (model_df["age"] < 18), "age"] = np.nan

print("Missing values per column before imputation:")
print(model_df.isnull().sum())

Missing values per column before imputation:
rule_triggered                               0
rule_assigned_severity_score               160
kyc_status                                   0
account_status                               0
premium_subscription_status                  0
age                                         25
province_region                              0
stated_risk_appetite                         0
stated_investment_goal                      65
transaction_type                          1061
amount                                    1061
portfolio_concentration_at_transaction    1885
target                                       0
dtype: int64


In [9]:
# Step 2e: Impute missing values (with reasoning per field)

# transaction_type / amount / concentration are missing where the alert was raised
# on ACCOUNT-LEVEL behaviour (no single transaction). This is meaningful and not an error.
# - transaction_type: fill with "None" as its own category (no transaction attached)
# - amount: fill with 0 (no transaction amount)
# - portfolio_concentration: fill with 0 (not applicable / no position)
model_df["transaction_type"] = model_df["transaction_type"].fillna("None")
model_df["amount"] = model_df["amount"].fillna(0)
model_df["portfolio_concentration_at_transaction"] = \
    model_df["portfolio_concentration_at_transaction"].fillna(0)

# rule_assigned_severity_score missing (~6%) where the engine emitted none.
# Fill with the median severity (a neutral middle value).
median_sev = model_df["rule_assigned_severity_score"].median()
model_df["rule_assigned_severity_score"] = \
    model_df["rule_assigned_severity_score"].fillna(median_sev)

# age missing (real gaps + the out-of-range values we nulled): fill with median age.
median_age = model_df["age"].median()
model_df["age"] = model_df["age"].fillna(median_age)

# stated_investment_goal missing: fill with "Unknown" as its own category.
model_df["stated_investment_goal"] = model_df["stated_investment_goal"].fillna("Unknown")

print("Missing values after imputation:", model_df.isnull().sum().sum())
print("Shape:", model_df.shape)

Missing values after imputation: 0
Shape: (2587, 13)


In [10]:
# Step 2f: Encode categorical features and separate X / y 
# Models need numeric input. We one-hot encode the categorical columns, which
# turns each category into its own 0/1 column without implying any false ordering.

# Separate the target
y = model_df["target"]
X = model_df.drop("target", axis=1)

# One-hot encode all categorical (text) columns
X = pd.get_dummies(X, drop_first=False)

print("Feature matrix shape after encoding:", X.shape)
print("Number of features:", X.shape[1])

Feature matrix shape after encoding: (2587, 47)
Number of features: 47


In [11]:
# Step 2g: Train/test split (stratified), then SMOTE on training only
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

# Stratified 80/20 split. Stratify on y so both sets keep the same
# True-positive / False-positive ratio, and a fixed seed for reproducibility.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Before SMOTE:")
print("  Training set:", X_train.shape, "| positives:", int(y_train.sum()))
print("  Test set:    ", X_test.shape, "| positives:", int(y_test.sum()))

# Apply SMOTE to the TRAINING SET ONLY, to balance the classes for training.
# The test set is left untouched so evaluation reflects the real class balance.
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print("\nAfter SMOTE (training set only):")
print("  Training set:", X_train_bal.shape)
print("  Class balance:", y_train_bal.value_counts().to_dict())

Before SMOTE:
  Training set: (2069, 47) | positives: 281
  Test set:     (518, 47) | positives: 70

After SMOTE (training set only):
  Training set: (3576, 47)
  Class balance: {1: 1788, 0: 1788}


### Preprocessing — explanation of choices

Preprocessing was necessary because unlike the clean SS2 proxy dataset, this is
an uncleaned extract from live systems. Each step below addresses a specific issue
and keeps the method aligned with SS2.

**1. Removed Unresolved alerts.** These 277 alerts carry no known outcome and
cannot serve as training labels, so they were removed, leaving a binary problem
(True positive vs False positive). This mirrors the SS2 approach of a clear,
labelled target.

**2. Cleaned categorical text.** The extract contained inconsistent casing and
stray whitespace (for example `'GP'`, `'gp'` and `' GP '` all appeared). These
were stripped and standardised so the same real value is not split into several
false categories, which would weaken the model.

**3. Joined features from multiple tables.** The alert table alone is
feature-poor. Account profile (table 2), demographics (table 3) and transaction
context (table 4) were joined on `account_id` and `transaction_id` using left
joins, so every alert is retained and enriched where data exists. This produces a
far richer feature set than the anonymised SS2 data allowed.

**4. Handled missing and out-of-range values with meaning.** Transaction fields
are missing where an alert was raised on account-level behaviour rather than a
single transaction; these were filled with a distinct "None" category and zero
amount, preserving that real meaning rather than discarding the alerts. Severity
and age were filled with the median, and implausible ages (over 100, such as the
recorded 173) were treated as missing first. Investment goal gaps became an
"Unknown" category.

**5. Dropped leakage-prone columns.** The `investigation_disposition` text and
`case_closed_timestamp` were excluded from the features. The disposition text is
the answer itself, and a closed-case timestamp only exists because a case was
resolved, so including either would leak the outcome and produce dishonestly high
scores.

**6. One-hot encoded categoricals.** Categorical features were one-hot encoded so
that unordered categories (such as province) are not given a false numeric
ordering, expanding the feature set to 47 columns.

**7. Stratified split, then SMOTE on the training set only.** The data was split
80/20 with stratification so both sets keep the real ~14% positive rate. SMOTE
was then applied to the **training set only**, balancing it to 50/50 so the model
learns both classes, while the test set was left at its real imbalance so that
validation reflects true performance. This is the exact imbalance-handling
strategy used in SS2, and applying SMOTE only to training avoids data leakage.